In [1]:
import pandas as pd
import os

raw_path = "../data/raw"
files = sorted([f for f in os.listdir(raw_path) if f.endswith(".csv")])

dfs = {}
for f in files:
    name = f.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    dfs[name] = pd.read_csv(os.path.join(raw_path, f))
    print(f"{name:25s} rows={len(dfs[name]):7,d}  cols={dfs[name].shape[1]}")

customers                 rows= 99,441  cols=5
geolocation               rows=1,000,163  cols=5
order_items               rows=112,650  cols=7
order_payments            rows=103,886  cols=5
order_reviews             rows= 99,224  cols=7
orders                    rows= 99,441  cols=8
products                  rows= 32,951  cols=9
sellers                   rows=  3,095  cols=4
product_category_name_translation rows=     71  cols=2


In [2]:
for name, df in dfs.items():
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(df.dtypes)
    print(df.head(3))


customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object
   geolocatio

In [3]:
for name, df in dfs.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n{name}:")
        print(missing)


order_reviews:
review_comment_title      87656
review_comment_message    58247
dtype: int64

orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [4]:
for name, df in dfs.items():
    dupes = df.duplicated().sum()
    print(f"{name:25s} full-row duplicates: {dupes}")

customers                 full-row duplicates: 0
geolocation               full-row duplicates: 261831
order_items               full-row duplicates: 0
order_payments            full-row duplicates: 0
order_reviews             full-row duplicates: 0
orders                    full-row duplicates: 0
products                  full-row duplicates: 0
sellers                   full-row duplicates: 0
product_category_name_translation full-row duplicates: 0


In [5]:
key_checks = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
    "sellers": "seller_id",
}

for name, col in key_checks.items():
    df = dfs[name]
    n_rows = len(df)
    n_unique = df[col].nunique()
    print(f"{name:12s} {col:15s} rows={n_rows:7,d}  unique={n_unique:7,d}  {'OK' if n_rows==n_unique else 'MISMATCH'}")

customers    customer_id     rows= 99,441  unique= 99,441  OK
orders       order_id        rows= 99,441  unique= 99,441  OK
products     product_id      rows= 32,951  unique= 32,951  OK
sellers      seller_id       rows=  3,095  unique=  3,095  OK


In [6]:
orders = dfs["orders"]
date_cols = [c for c in orders.columns if "date" in c or "timestamp" in c]
for c in date_cols:
    parsed = pd.to_datetime(orders[c], errors="coerce")
    print(f"{c:35s} min={parsed.min()}  max={parsed.max()}  nulls={parsed.isnull().sum()}")

order_purchase_timestamp            min=2016-09-04 21:15:19  max=2018-10-17 17:30:18  nulls=0
order_delivered_carrier_date        min=2016-10-08 10:34:01  max=2018-09-11 19:48:28  nulls=1783
order_delivered_customer_date       min=2016-10-11 13:46:32  max=2018-10-17 13:22:46  nulls=2965
order_estimated_delivery_date       min=2016-09-30 00:00:00  max=2018-11-12 00:00:00  nulls=0


In [7]:
print(dfs["orders"]["order_status"].value_counts())
print(dfs["order_payments"]["payment_type"].value_counts())
print(dfs["customers"]["customer_state"].value_counts().head(10))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64
